# 04 — Feature Selection
Rank features by importance, check multicollinearity, finalize feature set.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression

from src.features.build_features import build_features

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Ready.')

Ready.


## 1. Load clean data

In [2]:
df = pd.read_csv('../../data/processed/bioage_final_clean.csv')
print(f'Clean data: {df.shape}')
feature_cols = [c for c in df.columns if c not in ['Age', 'age_group']]
X = df[feature_cols].values
y = df['Age'].values

Clean data: (19992, 13)


## 2. Pearson correlation with Age

In [3]:
corr_with_age = df[feature_cols + ['Age']].corr()['Age'].drop('Age').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr_with_age.values]
ax.barh(corr_with_age.index, corr_with_age.values, color=colors)
ax.set_xlabel('Pearson r with Age')
ax.set_title('Feature–Age Correlation Ranking')
ax.axvline(x=0, color='black', linewidth=0.5)
plt.tight_layout()
plt.savefig('../../reports/figures/11_correlation_with_age.png', dpi=150)
plt.show()
print('Saved: reports/figures/11_correlation_with_age.png')

Saved: reports/figures/11_correlation_with_age.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_24160\1005732495.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Mutual Information with Age

In [4]:
mi = mutual_info_regression(X, y, random_state=42)
mi_series = pd.Series(mi, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(mi_series.index, mi_series.values, color='steelblue')
ax.set_xlabel('Mutual Information')
ax.set_title('Feature Importance — Mutual Information with Age')
plt.tight_layout()
plt.savefig('../../reports/figures/12_mutual_information.png', dpi=150)
plt.show()
print('Saved: reports/figures/12_mutual_information.png')

Saved: reports/figures/12_mutual_information.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_24160\253797330.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Multicollinearity check

In [5]:
corr_matrix = df[feature_cols].corr()

# Find pairs with |r| > 0.8
high_corr_pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.8:
            high_corr_pairs.append((feature_cols[i], feature_cols[j], r))

if high_corr_pairs:
    print('Highly correlated feature pairs (|r| > 0.8):')
    for f1, f2, r in sorted(high_corr_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {f1} <-> {f2}: r={r:.3f}')
else:
    print('No feature pairs with |r| > 0.8 found.')

Highly correlated feature pairs (|r| > 0.8):
  LBXGLU <-> LBXGH: r=0.832


In [6]:
# Clustered heatmap using scipy linkage (avoids fastcluster dependency)
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

condensed_dist = squareform(1 - corr_matrix.abs().values)
row_linkage = linkage(condensed_dist, method='average')
order = leaves_list(row_linkage)
ordered_cols = [corr_matrix.columns[i] for i in order]
ordered_corr = corr_matrix.loc[ordered_cols, ordered_cols]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(ordered_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation - Clustered Order')
plt.tight_layout()
plt.savefig('../../reports/figures/13_clustermap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/figures/13_clustermap.png')

Saved: reports/figures/13_clustermap.png


C:\Users\vinit\AppData\Local\Temp\ipykernel_24160\1734479234.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Final feature ranking

In [7]:
# Combined ranking: average of (correlation rank, MI rank)
corr_rank = corr_with_age.abs().rank(ascending=False)
mi_rank = mi_series.rank(ascending=False)

ranking = pd.DataFrame({
    'feature': feature_cols,
    'pearson_r': [corr_with_age.get(f, 0) for f in feature_cols],
    'mi_score': [mi_series.get(f, 0) for f in feature_cols],
    'corr_rank': [corr_rank.get(f, len(feature_cols)) for f in feature_cols],
    'mi_rank': [mi_rank.get(f, len(feature_cols)) for f in feature_cols],
})
ranking['avg_rank'] = (ranking['corr_rank'] + ranking['mi_rank']) / 2
ranking = ranking.sort_values('avg_rank')
ranking

,feature,pearson_r,mi_score,corr_rank,mi_rank,avg_rank
9,LBXGH,0.296605,0.162537,1.0,1.0,1.0
2,LBXGLU,0.232609,0.130855,3.0,2.0,2.5
1,LBXSCR,0.238509,0.101425,2.0,7.0,4.5
5,LBXMCVSI,0.204473,0.113757,4.0,6.0,5.0
4,LBXLYPCT,-0.174465,0.125355,6.0,4.0,5.0
6,LBXRDW,0.183826,0.068675,5.0,8.0,6.5
11,LBXTC,0.095963,0.114450,9.0,5.0,7.0
3,CRP,0.051221,0.126620,11.0,3.0,7.0
7,LBXSAPSI,0.130186,0.062307,8.0,9.0,8.5
0,LBXSAL,-0.164590,0.038166,7.0,11.0,9.0


## Recommended feature set
Based on correlation, mutual information, and multicollinearity analysis.
Highly correlated duplicates (|r| > 0.8) should have one removed.
Final feature list saved in the model-ready CSV.

In [8]:
# Build the model-ready dataset with log transforms
model_df = build_features(
    clean_path='../../data/processed/bioage_final_clean.csv',
    output_path='../../data/processed/bioage_model_ready.csv',
    log_transform_skewed=True,
    add_age_group=True,
)

Loaded clean data: 19992 rows x 13 cols
  Created log transform: log_CRP
  Created log transform: log_LBXSAPSI
  Created log transform: log_LBXWBCSI
  Created log transform: log_LBXGH
  Created age_group column with bins: ['0-17', '18-29', '30-44', '45-59', '60-74', '75+']
Saved model-ready data to ../../data/processed/bioage_model_ready.csv (19992 rows x 18 cols)
